In [ ]:
"""
Vector DBs
Vector DBs are used to store high-dimensional vectors and compare them to do
things like semantic (meaning) search as opposed to just keyword search

Today we'll be implementing this manually without a proper vector db

"""

import sqlite3
import json
import numpy as np
import pandas as pd
# This is what I've use to get our embeddings
from sentence_transformers import SentenceTransformer

In [ ]:
"""
Manual Cosine Similarity Function

As a reminder Cosine similarity checks the angle between vectors and returns
a value between -1 and 1 describing how "similar" they are
-1 being complete opposites
1 being the same

"""
def cosine_similarity(a, b):
  a = np.array(a, dtype=float)
  b = np.array(b, dtype=float)
  # We need to get the product of the vector norms
  denominator = np.linalg.norm(a) * np.linalg.norm(b)

  if denominator == 0.0:
    return 0.0

  return float(np.dot(a,b) / denominator)

query_vector = [1, 1]

examples ={
    "very similar": [.9, 1.1],
    "somewhat similar": [1, 0],
    "opposite": [-1, -1]
}

for label, vect in examples.items():
  print(label, cosine_similarity(query_vector, vect))

very similar 0.995037190209989
somewhat similar 0.7071067811865475
opposite -0.9999999999999998


In [ ]:
"""
Now that we can calculate how similar two vectors are, let's create
some "documents' that we can search through with semantic search

Vector DBs typically include the following columns:
id or document name
metadata (to gather info about the document or use it to help filter our results)
text
vector embedding
"""

documents = [
    {"id": "doc-001", "topic": "tensorflow", "text": "CNNs are useful for image classification tasks"},
    {"id": "doc-002", "topic": "tensorflow", "text": "Convolutional neural networks work well when analyzing pictures."},
    {"id": "doc-003", "topic": "ml", "text": "Cross-validation helps estimate how well a model generalizes."},
    {"id": "doc-004", "topic": "ml", "text": "Precision and recall are useful metrics for imbalanced classification."},
    {"id": "doc-005", "topic": "vectordb", "text": "Vector databases store embeddings for similarity search."},
    {"id": "doc-006", "topic": "vectordb", "text": "Chroma can run locally and store documents with metadata."},
    {"id": "doc-007", "topic": "aws", "text": "SageMaker is used to train and deploy machine learning models on AWS."},
    {"id": "doc-008", "topic": "langchain", "text": "RAG retrieves relevant context before sending a prompt to a language model."},
]

df = pd.DataFrame(documents)
df


,id,topic,text
0,doc-001,tensorflow,CNNs are useful for image classification tasks
1,doc-002,tensorflow,Convolutional neural networks work well when a...
2,doc-003,ml,Cross-validation helps estimate how well a mod...
3,doc-004,ml,Precision and recall are useful metrics for im...
4,doc-005,vectordb,Vector databases store embeddings for similari...
5,doc-006,vectordb,Chroma can run locally and store documents wit...
6,doc-007,aws,SageMaker is used to train and deploy machine ...
7,doc-008,langchain,RAG retrieves relevant context before sending ...


In [ ]:
"""
At this point we have all of our texts, we need to generate our embeddings
We could create a whole network for this but I'm going to use an existing model
"""

model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
# This is a sentence model that take a sentence, tokenizes, embeds and pools our sentence
# in a 384-dimension vector

texts = df["text"].tolist()
embeddings = model.encode(texts, normalize_embeddings=True)

print("Number of Documents:", len(embeddings))
print("Embedding Dimension:", len(embeddings[0]))

model.safetensors:   0%|          | 0.00/90.9M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Number of Documents: 8
Embedding Dimension: 384


In [ ]:
"""
Try this with a SQL table

Let's load all of this into our SQL table and go from there
"""

# Connection
conn = sqlite3.connect(":memory:") # Only exists in memory, will be deleted if rerun
# Cursor
cur = conn.cursor()

# Execute the Queries

# Create a table
cur.execute("""
CREATE TABLE documents(
  id TEXT PRIMARY KEY,
  topic TEXT,
  text TEXT,
  embedding_json TEXT
)
""")

# Let's add in our rows and embeddings
for row, embedding in zip(documents, embeddings):
  cur.execute(
      "INSERT INTO documents VALUES (?,?,?,?)",
      (row["id"], row["topic"], row["text"], json.dumps(embedding.tolist()))
  )

conn.commit()

pd.read_sql_query("SELECT * FROM documents", conn)

,id,topic,text,embedding_json
0,doc-001,tensorflow,CNNs are useful for image classification tasks,"[-0.10556063055992126, -0.034237537533044815, ..."
1,doc-002,tensorflow,Convolutional neural networks work well when a...,"[-0.03481575474143028, -0.04238727688789368, 0..."
2,doc-003,ml,Cross-validation helps estimate how well a mod...,"[-0.031431641429662704, -0.06941051781177521, ..."
3,doc-004,ml,Precision and recall are useful metrics for im...,"[-0.04966140165925026, -0.022200779989361763, ..."
4,doc-005,vectordb,Vector databases store embeddings for similari...,"[-0.017895642668008804, -0.07363148778676987, ..."
5,doc-006,vectordb,Chroma can run locally and store documents wit...,"[-0.04076027870178223, 0.042175062000751495, -..."
6,doc-007,aws,SageMaker is used to train and deploy machine ...,"[-0.08184503763914108, -0.07434552162885666, -..."
7,doc-008,langchain,RAG retrieves relevant context before sending ...,"[-0.01745324209332466, 0.021954314783215523, 0..."


In [ ]:
"""

Manual Vector Search (Keyword Search)

Why is using SQL for this not the preferred option? Well in SQL you can
only really do keyword search before the vector similarities so its a little
more difficult

"""

keyword = "pictures"

pd.read_sql_query(
    "SELECT id, topic, text FROM documents WHERE text LIKE ?",
    conn,
    params=(f"%{keyword}%",)
)

,id,topic,text
0,doc-002,tensorflow,Convolutional neural networks work well when a...


In [ ]:
"""
Define a new function for searching with vectors
"""
def sqlite_vector_search(query, top_k = 3, topic_filter=None):
  # Get an embedding of the query vector
  query_embedding = model.encode([query], normalize_embeddings=True)[0]

  # Search in a specific topic or else search all
  if topic_filter:
    rows = cur.execute(
        "SELECT * FROM documents WHERE topic = ?",
        (topic_filter, )
    ).fetchall()
  else:
    rows = cur.execute(
        "SELECT * FROM documents"
    ).fetchall()

  scored = []
  for doc_id, topic, text, embedding_json in rows:
    doc_embedding = np.array(json.loads(embedding_json))
    score = cosine_similarity(query_embedding, doc_embedding)

    scored.append({
        "id": doc_id,
        "topic": topic,
        "text": text,
        "score": score
    })

  return pd.DataFrame(scored).sort_values("score", ascending=False).head(top_k)

sqlite_vector_search("pictures")

,id,topic,text,score
1,doc-002,tensorflow,Convolutional neural networks work well when a...,0.424929
0,doc-001,tensorflow,CNNs are useful for image classification tasks,0.352316
4,doc-005,vectordb,Vector databases store embeddings for similari...,0.149912


In [ ]:
sqlite_vector_search("What neural network should I use for pictures?")

,id,topic,text,score
1,doc-002,tensorflow,Convolutional neural networks work well when a...,0.636517
0,doc-001,tensorflow,CNNs are useful for image classification tasks,0.573443
4,doc-005,vectordb,Vector databases store embeddings for similari...,0.169531


In [ ]:
sqlite_vector_search("How can I evaluate a classification model?")

,id,topic,text,score
2,doc-003,ml,Cross-validation helps estimate how well a mod...,0.407839
3,doc-004,ml,Precision and recall are useful metrics for im...,0.381965
0,doc-001,tensorflow,CNNs are useful for image classification tasks,0.205238


In [ ]:
sqlite_vector_search("How do I search by meaning?", topic_filter="vectordb")
# Using metadata to filter the documents you're looking through

,id,topic,text,score
0,doc-005,vectordb,Vector databases store embeddings for similari...,0.334334
1,doc-006,vectordb,Chroma can run locally and store documents wit...,0.079392


In [ ]:
sqlite_vector_search("How can I deploy a machine learning model?")

,id,topic,text,score
6,doc-007,aws,SageMaker is used to train and deploy machine ...,0.542203
2,doc-003,ml,Cross-validation helps estimate how well a mod...,0.235405
5,doc-006,vectordb,Chroma can run locally and store documents wit...,0.144729
